In [ ]:
from dotenv import load_dotenv

# set any relevant environment variables, such as HF_TOKEN,
# which speed up downloads of models
load_dotenv()

In [ ]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, HfApiModel

model = HfApiModel()
agent = CodeAgent(tools=[DuckDuckGoSearchTool()], model=model)
response = agent.run(
    "How long does it take to travel from " + "New York to Los Angeles by train?"
)
print(response)

In [ ]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, LiteLLMModel

model = LiteLLMModel(
    model_id="ollama/qwen2:7b",
    api_base="http://127.0.0.1:11434",
    num_ctx=8192,
)
agent = CodeAgent(tools=[DuckDuckGoSearchTool()], model=model)
response = agent.run(
    "How long does it take to travel from " + " New York to Los Angeles by train?"
)
print(response)

In [ ]:
import os

model = LiteLLMModel(
    model_id="gpt-4o-mini",
    api_base="https://api.openai.com/v1",
)

In [ ]:
from smolagents import CodeAgent, LiteLLMModel, PythonInterpreterTool

model = LiteLLMModel(
    model_id="gpt-4o-mini",
    api_base="https://api.openai.com/v1",
)

agent = CodeAgent(tools=[PythonInterpreterTool()], model=model)
response = agent.run("Calculate the 10th Fibonacci number.")
print(response)

In [ ]:
import requests
from smolagents import CodeAgent, LiteLLMModel, tool


@tool
def get_weather_info(city: str) -> str:
    """Retrieve the current weather information for a given city.
    Args:
        city: The name of the city to get the weather information for.
    Returns:
        str: A description of the current weather and temperature in
        the city.
    """
    api_key = os.getenv("OPENWEATHER_API_KEY")
    url = (
        f"http://api.openweathermap.org/data/2.5/weather?q={city}&"
        f"appid={api_key}&units=metric"
    )
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        weather = data["weather"][0]["description"]
        temperature = data["main"]["temp"]
        return (
            f"The weather in {city} is {weather} with a temperature of {temperature}°C."
        )
    else:
        return f"Could not retrieve weather information for {city}."


model = LiteLLMModel(
    model_id="ollama/qwen2:7b",
    api_base="http://127.0.0.1:11434",
    num_ctx=8192,
)

agent = CodeAgent(tools=[get_weather_info], model=model)
response = agent.run("What is the current weather for Singapore?")
print(response)

In [ ]:
from langchain.tools import Tool
from langchain_community.utilities.serpapi import SerpAPIWrapper

search = SerpAPIWrapper()
tools = [
    Tool(
        name="Search",
        func=search.run,
        description=(
            "Useful for when you need to answer questions about current events "
            "or search for specific information on the web. "
            "Input should be a search query."
        ),
    )
]

In [ ]:
from langchain.agents import AgentType, initialize_agent
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
agent = initialize_agent(
    tools=tools, llm=llm, agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

response = agent.invoke("Who is Wei-Meng Lee?")
print(response)

In [ ]:
response = agent.invoke("What is the weather in New York today?")
print(response)

In [ ]:
from langchain.agents import AgentType, initialize_agent
from langchain.tools import Tool, tool
from langchain_community.utilities.serpapi import SerpAPIWrapper
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(temperature=0)

search = SerpAPIWrapper()
search_tool = Tool(
    name="Search",
    func=search.run,
    description=(
        "Useful for when you need to answer questions about current events "
        "or search for specific information on the web. "
        "Input should be a search query."
    ),
)


@tool
def get_weather_info(city: str) -> str:
    """Retrieve the current weather information for a given city.
    Args:
        city: The name of the city to get the weather information for.
    Returns:
        str: A description of the current weather and temperature in
        the city.
    """

    api_key = os.getenv("OPENWEATHER_API_KEY")
    url = (
        f"http://api.openweathermap.org/data/2.5/weather?q={city}&"
        f"appid={api_key}&units=metric"
    )
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        weather = data["weather"][0]["description"]
        temperature = data["main"]["temp"]
        humidity = data["main"]["humidity"]
        wind_speed = data["wind"]["speed"]
        summary = (
            f"Weather in {city}:\n"
            f"Condition: {weather}\n"
            f"Temperature: {temperature}°C\n"
            f"Humidity: {humidity}%\n"
            f"Wind Speed: {wind_speed} m/s"
        )
        return summary
    else:
        return f"Could not retrieve weather information for {city}."


tools = [search_tool, get_weather_info]
agent = initialize_agent(
    tools=tools, llm=llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [ ]:
response = agent.invoke("What is the current weather of Singapore?")
print(response)

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages
from langgraph.prebuilt import create_react_agent

In [ ]:
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)
tools = []

In [ ]:
agent_executor = create_react_agent(llm, tools)

In [ ]:
def run_agent(query: str):
    state = agent_executor.invoke({"messages": [("user", query)]})
    print(state)
    print("\n📜 Agent trace:")
    for i, msg in enumerate(state["messages"]):
        print(
            f"{i + 1}. [{msg.type.upper()}] "
            f"{msg.content.strip() if hasattr(msg, 'content') else msg}"
        )
        print("=====")
    return state["messages"][-1].content

In [ ]:
query = "Who is Bill Gates?"
answer = run_agent(query)
print(f"Query: {query}")
print(f"Answer: {answer}")

In [ ]:
from langchain_community.utilities import SerpAPIWrapper
from langchain_core.tools import Tool

serpapi = SerpAPIWrapper()
search_tool = Tool(
    name="SerpAPI",
    func=serpapi.run,
    description="A search engine tool to query real-time information from the web.",
)

In [ ]:
tools = [search_tool]
agent_executor = create_react_agent(llm, tools)

In [ ]:
query = "Who won the US Presidential Election in 2024?"
answer = run_agent(query)
print(f"Query: {query}")
print(f"Answer: {answer}")

In [ ]:
# =================================================
# Define a custom tool to fetch weather information
# =================================================
def get_weather_info(city: str) -> str:
    """Retrieve the current weather information for a given city."""
    api_key = os.getenv("OPENWEATHER_API_KEY")
    url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        weather = data["weather"][0]["description"]
        temperature = data["main"]["temp"]
        humidity = data["main"]["humidity"]
        wind_speed = data["wind"]["speed"]
        summary = (
            f"Weather in {city}:\n"
            f"Condition: {weather}\n"
            f"Temperature: {temperature}°C\n"
            f"Humidity: {humidity}%\n"
            f"Wind Speed: {wind_speed} m/s"
        )
        return summary
    else:
        return f"Could not retrieve weather information for {city}."

In [ ]:
weather_tool = Tool(
    name="GetWeather",
    func=get_weather_info,
    description="A tool to fetch the weather information for a city",
)

In [ ]:
tools = [search_tool, weather_tool]
agent_executor = create_react_agent(llm, tools)

In [ ]:
query = "What is the current weather for Singapore"
answer = run_agent(query)
print(f"Query: {query}")
print(f"Answer: {answer}")

In [ ]:
from pprint import pprint
from typing import Annotated

from typing_extensions import TypedDict

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
def run_agent(query: str, state: State = None) -> tuple[str, State]:
    response = agent_executor.invoke({"messages": [("user", query)]})
    pprint(response)  # print out the response from the agent
    if state is None:
        state = {"messages": []}
    state["messages"].append(("user", query))
    response = agent_executor.invoke(state)
    state = {"messages": response["messages"]}
    return response["messages"][-1].content, state

In [ ]:
conversation_state = {"messages": []}
while True:
    query = input("Question: ")
    if query.lower() == "quit":
        break
    answer, conversation_state = run_agent(query, conversation_state)
    print(f"Query: {query}")
    print(f"Answer: {answer}")